# Data Preparation: Adult Income Census

Uses insights from EDA to preprocess the adult income dataset, split the data, and upload to Hugging Face.

In [ ]:
import sys
import os

sys.path.append(os.path.join(os.getcwd(), "../.."))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from src.preprocess import to_tensors
from src.data import save_to_hf

## Load Data

In [ ]:
df = pd.read_csv(
    "hf://datasets/scikit-learn/adult-census-income/adult.csv", na_values=["?"]
)
print(f"Raw dataset: {df.shape}")

## Preprocessing

### 1 · Handle Missing Values

In [ ]:
def fill_missing_with_mode(df, columns):
    """Fill missing values with mode for specified columns."""
    for col in columns:
        df[col] = df[col].fillna(df[col].value_counts().index[0])
    return df


df = fill_missing_with_mode(df, ["occupation", "workclass", "native.country"])
print(f"Missing values remaining: {df.isna().sum().sum()}")

### 2 · Drop Redundant Columns

In [ ]:
# Drop redundant/low-value columns
COLS_TO_DROP = ["education", "fnlwgt", "marital.status"]
df.drop(columns=COLS_TO_DROP, inplace=True)
print(f"After drop: {df.shape}")

### 3 · Encode Binary Features

In [ ]:
# sex: Female=1, Male=0
df["sex"] = (df["sex"] == "Female").astype(int)

# income (target): >50K=1, <=50K=0
df["income"] = (df["income"] == ">50K").astype(int)

### 4 · Consolidate Categorical Values

In [ ]:
# Consolidate workclass
df["workclass"] = df["workclass"].replace(
    {
        "Without-pay": "Other",
        "Never-worked": "Other",
        "State-gov": "Government",
        "Local-gov": "Government",
        "Federal-gov": "Government",
        "Self-emp-not-inc": "Self-employed",
        "Self-emp-inc": "Self-employed",
    }
)

# Consolidate race
df["race"] = df["race"].replace(
    {"Asian-Pac-Islander": "Other", "Amer-Indian-Eskimo": "Other"}
)

# Consolidate native.country
df.loc[df["native.country"] != "United-States", "native.country"] = "Other"

### 5 · One-Hot Encoding

In [ ]:
OHE_COLS = ["workclass", "occupation", "relationship", "race", "native.country"]

df = pd.get_dummies(df, columns=OHE_COLS, drop_first=True)

# Cast bool to int
bool_cols = df.select_dtypes("bool").columns
df[bool_cols] = df[bool_cols].astype(int)

print(f"After encoding: {df.shape}")

### 6 · Separate Features and Target

In [ ]:
X = df.drop(columns=["income"])
y = df[["income"]]

print(f"Features: {X.shape}, Target: {y.shape}")

## Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

## Transformation Pipeline

All transformations fit on train, applied to test to prevent data leakage.

### 1 · Standardize Numerical Features

In [ ]:
NUM_FEATURES = [
    "age",
    "education.num",
    "capital.gain",
    "capital.loss",
    "hours.per.week",
]

scaler = StandardScaler()
X_train[NUM_FEATURES] = scaler.fit_transform(X_train[NUM_FEATURES])
X_test[NUM_FEATURES] = scaler.transform(X_test[NUM_FEATURES])

print("Standardization complete.")

### 2 · Convert to PyTorch Tensors

In [ ]:
X_tr, y_tr = to_tensors(X_train.values, y_train.values.ravel())
X_te, y_te = to_tensors(X_test.values, y_test.values.ravel())

y_tr = y_tr.unsqueeze(1)  # (N,) → (N, 1) for BCE loss
y_te = y_te.unsqueeze(1)

print(f"Train tensors: X_tr={tuple(X_tr.shape)}, y_tr={tuple(y_tr.shape)}")
print(f"Test tensors: X_te={tuple(X_te.shape)}, y_te={tuple(y_te.shape)}")

## Upload to Hugging Face

In [ ]:
# Convert back to DataFrames for HF (save_to_hf expects DataFrames)
y_train_hf = y_train.reset_index(drop=True)
y_test_hf = y_test.reset_index(drop=True)

save_to_hf(
    X_train=X_train.reset_index(drop=True),
    X_test=X_test.reset_index(drop=True),
    y_train_log=y_train_hf["income"].values,
    y_test_log=y_test_hf["income"].values,
    repo_name="b-fatma/adult-income-census-federated",
    label_col="income",
)